# 항공대 문제해결기법 7주차

## 1번 - [트리의 지름(1967)](https://www.acmicpc.net/problem/1967)

## 풀이

생각보다 이 문제는 간단하다. 다음과 같은 특징을 이해한다면 금방 풀린다.

#### 임의의 노드에서 가장 거리가 먼 노드는 트리의 지름을 구성하는 두 노드 중 하나이다.

위의 말이 이해가 안되면, 백준 문제에 나온 예시 그림을 보면 이해가 될 것이다. 해당 그림에서 임의의 노드를 바탕으로 직접 탐색해보면 위의 말이 성립함을 알 수 있다.

그럼 다음과 같은 로직을 바탕으로 트리의 지름을 구성하는 두 노드의 길이를 구할 수 있다.
- i) 임의의 한 개의 노드로부터 거리가 가장 먼 노드까지 Search (DFS)
- ii) 거리가 가장 먼 노드를 기록하고 해당 위치로부터 거리가 가장 먼 노드까지 다시 Search (DFS)
- iii) 해당 길이가 트리의 지름이다.

## 전체 코드

In [ ]:
import sys

input = sys.stdin.readline
sys.setrecursionlimit(10**5)

N = int(input())

# N이 1인 경우 예외 처리
if N == 1:
    print(0)
    exit()

graph = [[] for _ in range(N + 1)]

for _ in range(N - 1):
    u, v, w = map(int, input().split())
    graph[u].append((v, w))
    graph[v].append((u, w))

#print(graph)

vis = [False for _ in range(N + 1)]
end_point = -int(1e9)
res = 0

def dfs(node, dist):
    global res, end_point, vis
    vis[node] = True

    for v, w in graph[node]:

        if vis[v]:
            continue
        
        if dist + w > res:
            res = dist + w
            end_point = v
        
        dfs(v, dist + w)

dfs(1, 0)

vis = [False for _ in range(N + 1)]
dfs(end_point, 0)
print(res)
    
        

## 2번 - [침투(13565)](https://www.acmicpc.net/problem/13565)

## 풀이

문제는 아주 간단하다. 맨 윗층에서부터 Flood-Fill 방식을 적용하고, 맨 아래에서 전류가 닿은 곳이 있는지 없는지 체크한 뒤 조건에 맞게 출력하면 된다.

## 전체 코드

In [ ]:
import sys
from collections import deque

input = sys.stdin.readline


# 전처리
M, N = map(int, input().split())

board = [input().strip() for _ in range(M)]
dx = [-1, 1, 0, 0]
dy = [0, 0, -1, 1]
vis = [[False for _ in range(N)] for _ in range(M)]

#print(board)


# BFS
def BFS(x, y):
    global board, dx, dy

    vis[x][y] = True

    q = deque()
    q.append((x, y))

    while q:
        cur_x, cur_y = q.popleft()

        for dir in range(4):
            nx, ny = (cur_x + dx[dir], cur_y + dy[dir])

            if nx < 0 or nx >= M or ny < 0 or ny >= N:
                continue
            if vis[nx][ny] or board[nx][ny] != '0':
                continue

            vis[nx][ny] = True
            q.append((nx, ny))


# Flood-Fill
for i in range(N):
    if board[0][i] == '0':
        BFS(0, i)

for i in range(N):
    if vis[M - 1][i]:
        print("YES")
        exit()

print("NO")
        



## 3번 - [최솟값(10868)](https://www.acmicpc.net/problem/10868)

## 풀이

이 문제를 나이브하게 접근하면 무조건 시간초과에 걸린다. 그렇기 때문에 우린 수업에서 배운 방식대로 접근해야 하는데, 나는 세그먼트 트리로 접근하려고 한다.

### 1. 필요성
- **문제 상황**: 데이터가 빈번하게 변경되는 상황에서 특정 구간의 정보(합, 최솟값, 최댓값 등)를 빠르게 구해야 할 때 사용한다.
- **나이브(Naive) 방식의 한계**:
    - Query: 매번 구간을 순회하면 $O(n)$이 소요된다.
    - Update: 배열 값을 바꾸는 것은 $O(1)$이지만, Query 속도가 너무 느려 전체 성능이 저하된다.
- **세그먼트 트리의 위력**: Query와 Update을 모두 $O(\log n)$에 해결하는 "부분적 재빌드(Partial Re-build)" 구조를 가진다.

---

### 2. 자료구조의 원리
- **구조**: 전체 범위를 반씩 쪼개어 관리하는 **완전 이진 트리** 형태이다.
- **노드 구성**:
    - **리프 노드**: 배열의 실제 데이터들이 저장된다.
    - **부모 노드**: 자식 노드들의 결과값(예: `min(left, right)`)을 미리 계산하여 저장한다.
- **공간 복잡도**: 인덱스 에러 방지를 위해 보통 **$4N$** 크기의 배열을 할당한다.

---

### 3. 핵심 로직 (RMQ 기준)

#### ① Build (생성)
- 전체 범위를 절반씩 나누어 리프 노드까지 내려간 뒤, 올라오면서 부모 노드에 자식들의 최솟값을 채운다.
- **시간 복잡도**: $O(n)$

#### ② Query (조회)
- 찾고자 하는 구간이 노드 범위에 완전히 포함되면 즉시 값을 반환하고, 전혀 겹치지 않으면 무시(inf 반환), 걸쳐 있으면 자식에게 물어본다.
- **시간 복잡도**: $O(\log n)$

#### ③ Update (갱신)
- 특정 데이터가 바뀌면 해당 데이터를 포함하는 리프부터 루트까지의 경로(Path)에 있는 노드들만 다시 계산한다. (사실상의 부분적 Re-build)
- **시간 복잡도**: $O(\log n)$

이 개념을 바탕으로 전체 코드를 살펴보면 이해할 수 있을 것이다.

## 전체 코드

In [ ]:
import sys


# 전처리
sys.setrecursionlimit(10**5)

input = sys.stdin.readline

N, M = map(int, input().split())
tree = [0] * (4 * N)
data = [int(input()) for _ in range(N)]


# 세그먼트 트리 build - update - query
def build(node, start, end):
    global tree, data

    if start == end:
        tree[node] = data[start]
        return tree[node]
    
    mid = (start + end) // 2
    tree[node] = min(build(node*2, start, mid), build(node*2 + 1, mid + 1, end))
    return tree[node]


## update()는 지금 문제에선 필요 없음.
# def update(node, start, end, idx, val):
#     global tree, data

#     if idx < start or idx > end:
#         return
#     if start == end:
#         tree[node] = val
#         return
#     mid = (start + end) // 2
#     update(node*2, start, mid, idx, val)
#     update(node*2 + 1, mid + 1, end, idx, val)
#     tree[node] = min(tree[node*2], tree[node*2 + 1])


def query(node, start, end, left, right):
    global tree, data

    if left > end or right < start:
        return float('inf')
    if left <= start and end <= right:
        return tree[node]
    mid = (start + end) // 2
    return min(query(node*2, start, mid, left, right),
               query(node*2 + 1, mid + 1, end, left, right))


# solve

build(1, 0, N - 1)

for _ in range(M):
    a, b = map(int, input().split())
    ret = query(1, 0, N - 1, a - 1, b - 1)
    print(ret)

## 4번 - [수열과 쿼리 16(14428)](https://www.acmicpc.net/problem/14428)

## 풀이

3번 문제를 풀었으면, 4번 문제는 거기에서 응용한 것이다. tree의 값은 [idx, value] 형태의 리스트를 가지게 한 뒤, value만 가지고 최솟값을 정할 것이기에 min을 구하는 커스텀 함수 get_min()을 만든다.

이거 말곤 달라진 것이 없다. 아래 전체 코드를 봐보도록 하자.

## 전체 코드

In [ ]:
import sys
sys.setrecursionlimit(10**5)
input = sys.stdin.readline

N = int(input())
# 1. 리스트 컴프리헨션으로 독립된 객체 생성
tree = [[0, 0] for _ in range(4 * N + 1)]

# 2. data를 0-indexed로 깔끔하게 입력받기
A = list(map(int, input().split()))
data = []
for i in range(N):
    data.append([i + 1, A[i]]) # [인덱스, 값] 저장

# 최솟값 비교 함수: 값이 작으면 선택, 같으면 인덱스 작은 것 선택
def get_min(a, b):
    if a[1] < b[1]: return a
    elif a[1] > b[1]: return b
    else: return a if a[0] < b[0] else b

def build(node, start, end):
    if start == end:
        tree[node] = data[start]
        return tree[node]
    mid = (start + end) // 2
    tree[node] = get_min(build(node*2, start, mid), build(node*2 + 1, mid + 1, end))
    return tree[node]

def update(node, start, end, idx, val):
    if idx < start or idx > end:
        return tree[node]
    if start == end:
        tree[node] = [idx + 1, val] # 문제의 출력은 1-based 인덱스
        return tree[node]
    mid = (start + end) // 2
    tree[node] = get_min(update(node*2, start, mid, idx, val), 
                         update(node*2 + 1, mid + 1, end, idx, val))
    return tree[node]

def query(node, start, end, left, right):
    if left > end or right < start:
        return [float('inf'), float('inf')]
    if left <= start and end <= right:
        return tree[node]
    mid = (start + end) // 2
    return get_min(query(node*2, start, mid, left, right),
                   query(node*2 + 1, mid + 1, end, left, right))

build(1, 0, N - 1)

M = int(input())
for _ in range(M):
    order, x, y = map(int, input().split())
    if order == 1:
        # x번째 수를 y로 바꿈 (x는 1-based이므로 0-based로 보정)
        update(1, 0, N - 1, x - 1, y)
    else:
        # x번부터 y번까지 중 최솟값의 인덱스 출력
        print(query(1, 0, N - 1, x - 1, y - 1)[0])